# Drift-Sense — Hybrid ML Candidate Ranker (training notebook)

The core localizer is a **classical** template-matching + verification pipeline. This notebook trains the *optional* **hybrid** component: a small neural network (MLP) that scores each classical candidate's probability of being the true revisited site, replacing the hand-tuned three-regime selector.

**Design:** the MLP consumes the *same* per-candidate features the classical reliability logic uses (`candidate_features` in `localize.py`), so there is **no train/serve skew**. After training we export the weights to `ml_ranker.npz`; inference in `localize.py` is a **pure-numpy forward pass** (no sklearn/torch at run time). If the model file is absent, `localize` falls back to the classical selector — the pipeline always works.

**Data:** `ranker_data.npz`, built by `make_ranker_data.py` — 200 training + 40 test pairs, each run through the classical candidate generator; a candidate is labelled `1` iff it is within 5 px of ground truth. Pairs are split by group so none leaks across train/test.

In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from localize import FEATURE_ORDER

d = np.load('ranker_data.npz')
X_train, y_train, g_train = d['X_train'], d['y_train'], d['g_train']
X_test,  y_test,  g_test  = d['X_test'],  d['y_test'],  d['g_test']
print('train rows', X_train.shape, 'positives %.1f%%' % (100*y_train.mean()))
print('test  rows', X_test.shape,  'positives %.1f%%' % (100*y_test.mean()))
print('features:', FEATURE_ORDER)

## 1. Standardize + train the MLP
A deliberately small network `(16, 8)` with L2 regularization and early stopping — the feature space is low-dimensional and we want to generalize to the *unseen* official test set, not overfit 200 pairs.

In [ ]:
scaler = StandardScaler().fit(X_train)
clf = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu',
                    alpha=1e-3, max_iter=3000, early_stopping=True,
                    n_iter_no_change=30, random_state=0)
clf.fit(scaler.transform(X_train), y_train)
print('converged in', clf.n_iter_, 'iterations')

## 2. Evaluate
Per-candidate ROC-AUC, and the metric that actually matters — **pair-level selection accuracy**: does the highest-probability candidate in a pair correspond to the true site?

In [ ]:
p_train = clf.predict_proba(scaler.transform(X_train))[:, 1]
p_test  = clf.predict_proba(scaler.transform(X_test))[:, 1]
print('per-candidate ROC-AUC:  train %.3f   test %.3f'
      % (roc_auc_score(y_train, p_train), roc_auc_score(y_test, p_test)))

def pair_selection_accuracy(probs, y, g):
    solvable = hit_solvable = total = hit = 0
    for gid in np.unique(g):
        m = g == gid; pg, yg = probs[m], y[m]
        picked_true = yg[np.argmax(pg)] >= 1
        total += 1; hit += picked_true
        if yg.max() >= 1:
            solvable += 1; hit_solvable += picked_true
    return hit/total, hit_solvable/max(solvable,1), solvable, total

acc, acc_s, solv, tot = pair_selection_accuracy(p_test, y_test, g_test)
print('pair-level selection (test): %.1f%% of all %d pairs, %.1f%% of %d solvable pairs'
      % (100*acc, tot, 100*acc_s, solv))

In [ ]:
# Which features the model leans on (|input-layer weight| aggregated)
w0 = np.abs(clf.coefs_[0]).sum(axis=1)
for i in np.argsort(-w0)[:8]:
    print(f'{FEATURE_ORDER[i]:<18} {w0[i]:.2f}')

## 3. Export weights for pure-numpy inference
`localize.py` reloads these and runs the forward pass with numpy only — no sklearn dependency at inference.

In [ ]:
np.savez('ml_ranker.npz',
         mean=scaler.mean_, scale=scaler.scale_,
         n_layers=len(clf.coefs_),
         **{f'coef_{i}': c for i, c in enumerate(clf.coefs_)},
         **{f'intercept_{i}': b for i, b in enumerate(clf.intercepts_)})
print('exported ml_ranker.npz')

## 4. Head-to-head: classical vs hybrid on a fresh held-out set
Regenerates 30 unseen pairs and runs `localize` both ways (`use_ml=False` vs `use_ml=True`) to confirm the hybrid does not regress the classical accuracy.

In [ ]:
import cv2
from dataset_gen import generate_pair
from localize import localize

rng = np.random.default_rng(7)
e_cls, e_ml = [], []
for i, seed in enumerate(range(4000, 4030)):
    forced = rng.random() < 0.2
    ref, search, meta = generate_pair(i, seed, forced_periodic=forced)
    xc, yc, _ = localize(ref, search, use_ml=False)
    xm, ym, _ = localize(ref, search, use_ml=True)
    e_cls.append(np.hypot(xc-meta.gt_x, yc-meta.gt_y))
    e_ml.append(np.hypot(xm-meta.gt_x, ym-meta.gt_y))
e_cls, e_ml = np.array(e_cls), np.array(e_ml)
for name, e in [('classical', e_cls), ('hybrid  ', e_ml)]:
    print('%s  median %.2f  mean %5.1f  <1um %.0f%%  catastrophic(>100px) %d'
          % (name, np.median(e), e.mean(), 100*(e<100).mean(), (e>100).sum()))